# 03. Model Training

???????? ? ????????? ????????? ??????? ????????? ????????.

In [ ]:
import pandas as pd
import numpy as np
import yaml
import joblib
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.feature_extraction.text import TfidfVectorizer
from xgboost import XGBClassifier
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, 
    f1_score, roc_auc_score, confusion_matrix,
    classification_report, roc_curve
)
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

## ???????? ??????

In [ ]:
manual_features_df = pd.read_csv('features_data/manual_features.csv')
text_data = pd.read_csv('features_data/text_data.csv')['text']
labels = pd.read_csv('features_data/labels.csv')['label']

print(f"Manual features shape: {manual_features_df.shape}")
print(f"Text data shape: {text_data.shape}")
print(f"Labels shape: {labels.shape}")

## ?????????? ??????

In [ ]:
X_manual_train, X_manual_test, text_train, text_test, y_train, y_test = train_test_split(
    manual_features_df, text_data, labels, test_size=0.2, random_state=42, stratify=labels
)

tfidf = TfidfVectorizer(max_features=1000, ngram_range=(1, 2), min_df=2, max_df=0.95)
X_tfidf_train = tfidf.fit_transform(text_train)
X_tfidf_test = tfidf.transform(text_test)

tfidf_train_df = pd.DataFrame(
    X_tfidf_train.toarray(),
    columns=[f'tfidf_{i}' for i in range(X_tfidf_train.shape[1])],
    index=X_manual_train.index
)
tfidf_test_df = pd.DataFrame(
    X_tfidf_test.toarray(),
    columns=[f'tfidf_{i}' for i in range(X_tfidf_test.shape[1])],
    index=X_manual_test.index
)

X_train = pd.concat([X_manual_train, tfidf_train_df], axis=1)
X_test = pd.concat([X_manual_test, tfidf_test_df], axis=1)
feature_names = X_train.columns

print(f"Train size: {X_train.shape[0]}")
print(f"Test size: {X_test.shape[0]}")
print(f"TF-IDF feature count: {X_tfidf_train.shape[1]}")
print(f"Total feature count: {X_train.shape[1]}")
print(f"\nTrain class distribution:\n{y_train.value_counts()}")
print(f"\nTest class distribution:\n{y_test.value_counts()}")

## ???????? ???????

In [ ]:
models = {
    'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1),
    'Gradient Boosting': GradientBoostingClassifier(n_estimators=100, random_state=42),
    'XGBoost': XGBClassifier(n_estimators=100, random_state=42, use_label_encoder=False, eval_metric='logloss'),
}

results = {}

for name, model in models.items():
    print(f"\nTraining {name}...")
    model.fit(X_train, y_train)
    
    y_pred = model.predict(X_test)
    y_proba = model.predict_proba(X_test)[:, 1]
    
    results[name] = {
        'model': model,
        'accuracy': accuracy_score(y_test, y_pred),
        'precision': precision_score(y_test, y_pred),
        'recall': recall_score(y_test, y_pred),
        'f1': f1_score(y_test, y_pred),
        'roc_auc': roc_auc_score(y_test, y_proba),
        'y_pred': y_pred,
        'y_proba': y_proba,
    }
    
    print(f"  Accuracy: {results[name]['accuracy']:.4f}")
    print(f"  ROC-AUC: {results[name]['roc_auc']:.4f}")
    print(f"  F1-score: {results[name]['f1']:.4f}")

## ????????? ???????

In [ ]:
metrics_df = pd.DataFrame({
    'Model': results.keys(),
    'Accuracy': [r['accuracy'] for r in results.values()],
    'Precision': [r['precision'] for r in results.values()],
    'Recall': [r['recall'] for r in results.values()],
    'F1': [r['f1'] for r in results.values()],
    'ROC-AUC': [r['roc_auc'] for r in results.values()],
}).sort_values('F1', ascending=False)

print("Model Comparison:")
print(metrics_df.to_string(index=False))

## ???????????? ???????????

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

metrics_df_melted = metrics_df.melt(id_vars='Model', var_name='Metric', value_name='Score')
sns.barplot(data=metrics_df_melted, x='Model', y='Score', hue='Metric', ax=axes[0])
axes[0].set_title('????????? ?????? ???????')
axes[0].set_xticklabels(axes[0].get_xticklabels(), rotation=45, ha='right')
axes[0].legend(loc='upper right')

best_model_name = metrics_df.iloc[0]['Model']
fpr, tpr, _ = roc_curve(y_test, results[best_model_name]['y_proba'])
axes[1].plot(fpr, tpr, label=f'{best_model_name} (AUC = {results[best_model_name]["roc_auc"]:.3f})')
axes[1].plot([0, 1], [0, 1], 'k--')
axes[1].set_xlabel('False Positive Rate')
axes[1].set_ylabel('True Positive Rate')
axes[1].set_title('ROC Curve')
axes[1].legend()

plt.tight_layout()
plt.show()

## Confusion Matrix ?????? ??????

In [ ]:
best_model = results[best_model_name]['model']
y_pred_best = results[best_model_name]['y_pred']

cm = confusion_matrix(y_test, y_pred_best)

plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=['Legitimate', 'Phishing'],
            yticklabels=['Legitimate', 'Phishing'])
plt.title(f'Confusion Matrix - {best_model_name}')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.show()

print(f"\nClassification Report - {best_model_name}:")
print(classification_report(y_test, y_pred_best, target_names=['Legitimate', 'Phishing']))

## ???????? ????????? (??? Random Forest)

In [ ]:
rf_model = results['Random Forest']['model']
feature_importance = pd.DataFrame({
    'feature': feature_names,
    'importance': rf_model.feature_importances_
}).sort_values('importance', ascending=False).head(20)

plt.figure(figsize=(10, 8))
sns.barplot(data=feature_importance, x='importance', y='feature')
plt.title('Top 20 Feature Importance (Random Forest)')
plt.xlabel('Importance')
plt.ylabel('Feature')
plt.show()

## ?????????? ?????? ??????

In [ ]:
best_model = results[best_model_name]['model']
joblib.dump(best_model, 'models/best_model.pkl')
print(f"Best model ({best_model_name}) saved to models/best_model.pkl")